<a href="https://colab.research.google.com/github/bireswarchaudhuri-or/diamond-price-prediction-ml/blob/main/Diamond_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center>

# `LINEAR REGRESSION - Diamond ◇ Price Prediction`

</center>


## Assumptions of the Classical Linear Regression Model
---
### 1. Linearity

- The regression model is linear in the parameters; it may or may not be linear in the variables.
- This can be expressed as: $
  Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_k X_k + \varepsilon
  $



### 2. Independence

- The observations are independent of each other.
- This implies that the value of the dependent variable for one observation is not influenced by the value of the dependent variable for another observation.



### 3. Conditional Mean of Errors

- Given the value of $X_i$, the expected, or mean, value of the disturbance term $\varepsilon_i$ is zero.
- Mathematically, $ E(\varepsilon_i \mid X_i) = 0 $

### 4. Homoscedasticity

- The variance of each $\varepsilon_i$ is constant, or homoscedastic  
  (homo means equal and scedastic means variance).
- Mathematically,$ \operatorname{Var}(\varepsilon_i) = \sigma^2 \; \forall  i $
### 5. No Autocorrelation

- There is no correlation between two error terms.
- Mathematically, $
  \operatorname{Cov}(\varepsilon_i, \varepsilon_j) = 0 \quad \text{for } i \neq j
  $

### 6. No Perfect Multicollinearity

- The independent variables are not perfectly linearly related.
- This means that no independent variable is a perfect linear function of one or more other independent variables.


### 7. Exogeneity

- The independent variables are uncorrelated with the error term.
- This can be expressed as $
  \operatorname{Cov}(X_i, \varepsilon_i) = 0 \quad \text{for all } i
  $


### 8. Normality of Errors

- For hypothesis testing, the error term $\varepsilon$ follows the normal distribution with mean zero and (homoscedastic) variance.
- Mathematically, $
  \varepsilon_i \sim \mathcal{N}(0, \sigma^2)
  $

---------

######By ensuring these assumptions hold, the linear regression model provides **unbiased, consistent, and efficient** estimates of the coefficients$
(\beta_0, \beta_1, \ldots, \beta_k)
$


##Problem Definition
 - Analysis the price of Diamond by their attributes using Linear Regression

##Data
 - The Diamond dataset is a popular dataset provided by `kaggle` that contains data on Diamonds.
 - It includes various features related to the properties and their surroundings, along with the target variable, which is the Price of Diamond.

##Dataset Description
This classic dataset contains the prices and other attributes of almost 54,000 diamonds.Below is a detailed description of each feature:

1.   `Carat` :  Weight of the diamond
2.   `Cut` : Quality of  cut (Fair, Good, Very Good, Premium, Ideal)
3. `Color` : Diamond colour, from J (worst) to D (best)
4. ` Clarity` : Measurement of clearness the diamond (I1 (worst), SI2, SI1, VS2, VS1, VVS2, VVS1, IF (best))
5. `x` : Length in mm (0--10.74)
6. `y` : Width in mm (0--58.9)
7. `z` : Depth in mm (0--31.8)
8. `Depth`: Total depth percentage = z / mean(x, y) = 2 * z / (x + y) (43--79)
9. `table`: Width of top of diamond relative to widest point (43--95)

The Target Variable is :
- `Price` : Price of the Diamond in US Dollar($)



## 1. File Loading

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
path = "/content/drive/MyDrive/Colab Notebooks/diamonds.csv"
df = pd.read_csv(path)
df.head()

In [ ]:
df = df.drop(columns=['Unnamed: 0']) # Correctly drop the 'Unnamed: 0' column
df.head()

In [ ]:
df.shape

## 2. Import Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import scipy.stats as stats

## 3. Missing Data Analysis

> If the missing values are not handled properly we may end up drawing an inaccurate inference about the data. Due to improper handling, the result obtained will differ from the ones where missing values are present.



In [ ]:
# Printing total number of missing data
df.isnull().sum().sort_values(ascending=False)

# 4. Data Cleaning

> Rows where any of the diamond dimensions (x, y, z) were zero were removed, as a diamond cannot physically have zero length, width, or depth. These values were treated as measurement errors, and removing them improved data quality before feature engineering.



In [ ]:
# Retain those data which has no zero attribute
df = df[(df["x"] > 0) & (df["y"] > 0) & (df["z"] > 0)]

In [ ]:
df.shape

## 5. Pandas Profiling Report

In [ ]:
!pip install ydata-profiling

In [ ]:
# create Profiling Report
from ydata_profiling import ProfileReport
pf = ProfileReport(df, explorative = True, title="Pandas Profile Report")
pf.to_file('Output.html') # To save this report

# 6. Data Preprocessing
##6.1 Feature Engineering
###6.1.1 Ordinal-Encoder

>Machine Learning models cannot understand string values directly,
but they can process numerical data. Therefore, categorical features
such as
<code style="background:#2D1B3D; color:#F9A8D4; padding:2px 6px; border-radius:4px;">cut</code>,
<code style="background:#2D1B3D; color:#F9A8D4; padding:2px 6px; border-radius:4px;">color</code>, and
<code style="background:#2D1B3D; color:#F9A8D4; padding:2px 6px; border-radius:4px;">clarity</code>
must be converted into numerical form.
When such categorical features follow a meaningful order,
<b>Ordinal Encoding</b> is used to map categories into ordered numerical values.
<br><br>

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_features = ["cut", "color", "clarity"]
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order = ["J", "I", "H", "G", "F", "E", "D"]
clarity_order = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF" ]
oe = OrdinalEncoder(categories=[cut_order, color_order, clarity_order])
df[ordinal_features] = oe.fit_transform(df[ordinal_features])
df.head()

### 6.1.2 Yeo-Johnson Power Transformation
- Power transforms are a family of parametric, monotonic
transformations that are applied to make data more
Gaussian-like. This is useful for modeling issues related to `heteroscedasticity`, or other situations where `normality` is desired.
- The optimal parameter for stabilizing variance and minimizing skewness is estimated through `maximum likelihood`.
- Unlike the Box-Cox transformation, Yeo-Johnson Power Transformation can handle both positive and negative values.
- By default, **zero mean**, **unit-variance normalization** is applied to the transformed data.

In [ ]:
from sklearn.preprocessing import PowerTransformer

pt1 = PowerTransformer(method='yeo-johnson')
df = pt1.fit_transform(df)

The `lambda` values for the Yeo-Johnson transformation are parameters that optimize the transformation to best approximate a normal distribution for the given data.

In [ ]:
pt1.lambdas_

In [ ]:
# Converting to Pandas DataFrame
df = pd.DataFrame(df)
df.head()

### 6.1.3 Handling Outlier
#### 6.1.3.1 Visualization using Boxplot

In [ ]:
fig,ax = plt.subplots(figsize=(15,8))
sns.boxplot(data=df, ax=ax)


#### 6.1.3.2 Using Statistical Methods
**Replacing Outliers with Upper/Lower Bound Using IQR:**

Replacing outliers with upper/lower bounds using IQR involves calculating the interquartile range (IQR) of a numerical column. Outliers are values above Q3 + 1.5 * IQR or below Q1 - 1.5 * IQR. These outliers are then replaced with the nearest boundary (either Q3 + 1.5 * IQR or Q1 - 1.5 * IQR).

In [ ]:
for column in df.columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    UL = Q3 + 1.5 * IQR
    LL = Q1 - 1.5 * IQR
    df[column] = np.where(df[column] > UL, UL, df[column])
    df[column] = np.where(df[column] < LL, LL, df[column])

In [ ]:
fig,ax = plt.subplots(figsize=(15,8))
sns.boxplot(data=df, ax=ax)

## 6.1.4 Handling Skewness
*   If skewness is less than -1 or greater than +1, the distribution is highly skewed.
*    If skewness is between -1 and -1/2 or between +1/2 and +1, the distribution is moderately skewed.
* If skewness is between -1/2 and +1/2, the distribution is approximately skewed.


### 6.1.4.1 Visualization using Histogram

In [ ]:
df.hist(figsize=(15,8),bins=50,grid=False)

### 6.1.4.2 Calculating Skewness using `skew`

In [ ]:
from scipy.stats import skew
# Calculate skewness
skewness = df.apply(skew)
print("Skewness of Features:\n", skewness)

In [ ]:
# Identify moderately skewed columns
mod_skewed_columns = skewness[(skewness > -1/2) & (skewness < 1/2)].index
print("Moderately Skewed features : \n", mod_skewed_columns)

### 6.1.5 Handling Multicollinearity

> 1. Multicollinearity occurs when two or more predictor variables in a regression model are highly correlated with each other.



> 2. The Variance Inflation Factor (`VIF`) quantifies how much the variance of the coefficient estimates is inflated due to multicollinearity.

$$
\mathrm{VIF} = \frac{1}{1 - R_i^2}
$$


>3. A thumb rule for interpreting the Variance Inflation Factor:
> - **1** = Not correlated  
> - **Between 1 and 5** = Moderately correlated  
> - **Greater than 5** = Highly correlated  







#### 6.1.5.1 Correlation Plot

In [ ]:
corr=df.corr()
print(corr)
plt.figure(figsize=(8,8))

sns.heatmap(corr,
           cmap='YlGnBu',vmax=1.0, vmin=-1.0, linewidths=0.1,
           annot=True,square=True);
plt.title('Correlation between features')

# 0.0 - 0.2 -> Weak Correlation
# 0.3 - 0.6 -> Moderate Correlation
# 0.7 - 1.0 -> Strong Correlation
# 1 perfect positive
# -1 perfect negative

#### 6.1.5.2 Variance Inflation Factor(`VIF`)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
df.shape

In [ ]:
vif_df = pd.DataFrame()
vif_df['Features'] = df.columns
vif_df['VIF'] = [variance_inflation_factor(df,i) for i in range (df.shape[1])]
vif_df

### 6.1.5.3 Creating Volume to Reduce Multicollinearity
---
After calculating the Variance Inflation Factor (VIF), severe multicollinearity was observed among the size-related features `carat`, `x`, `y`, and `z`.

To address this:

- The variables `x`, `y`, and `z` represent the physical dimensions of a diamond and are highly correlated with each other and with `carat`.
- A new feature **`volume`** was created to combine dimensional information:$$ \text{volume} = x \times y \times z $$

- The original features `x`, `y`, and `z` were removed to reduce redundancy.
- This transformation preserves size information while lowering multicollinearity.
- The resulting feature set improves model stability and interpretability.

In [ ]:
df["volume"] = df[7] * df[8] * df[9]
df = df.drop(columns=[7,8,9])
df.head()

In [ ]:
vif_df = pd.DataFrame() # Re-initialize vif_df
vif_df['Features'] = df.columns
vif_df['VIF'] = [variance_inflation_factor(df,i) for i in range (df.shape[1])]
vif_df

After creating the volume feature, a high Variance Inflation Factor (VIF) was observed between carat and volume, indicating strong multicollinearity. Since carat is the most important predictor of diamond price and already captures size information effectively, the volume feature have been removed to avoid redundancy while preserving interpretability.



In [ ]:
df = df.drop(columns=['volume'])

In [ ]:
vif_df = pd.DataFrame() # Re-initialize vif_df
vif_df['Features'] = df.columns
vif_df['VIF'] = [variance_inflation_factor(df,i) for i in range (df.shape[1])]
vif_df

Although the carat feature showed a relatively high VIF, it was retained because it is the most important predictor of diamond price and has clear physical meaning. Removing carat significantly reduces the predictive power of the model, so it was preserved while redundant size-related variables were removed.

# 7. Splitting the dataset into Train & Test

In [ ]:
y = df[6]
x = df.drop(columns=[6])

In [ ]:
# Let us split the dataset into two parts
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state = 0)

In [ ]:
print("X-Train data :", x_train.shape)
print("X-Test data :", x_test.shape)
print("Y-Train data :", y_train.shape)
print("Y-Test data :", y_test.shape)

# 8. Linear Regression Modelling
## 8.1. Building a Linear Regressssion Model

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()

In [ ]:
# Fit Linear Regression Model
model.fit(x_train,y_train)

In [ ]:
# prediction
y_pred = model.predict(x_test)

## 8.2 Model Evaluation

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

### 8.2.1 R-squared

R-squared $R^2$ measures the proportion of variance in the target variable that can be explained by the independent variables in a regression model. It indicates how well the model fits the data.

The formula for $R^2$ is:

$$
R^2 = 1 - \frac{SS_{res}}{SS_{tot}}
$$

Where:

- $SS_{res}$ (Residual Sum of Squares) represents the unexplained variance.
- $SS_{tot}$ (Total Sum of Squares) represents the total variance in the observed data.

A higher $R^2$ value indicates a better fit of the model to the data.

In [ ]:
r2 = r2_score(y_test, y_pred)
print("R-squared:", r2)

### 8.2.2 Mean Absolute Error (MAE)

Mean Absolute Error (MAE) is a regression evaluation metric that measures the average absolute difference between actual and predicted values. It provides a straightforward interpretation of prediction error in the same units as the target variable.

The formula for MAE is:

$$
\mathrm{MAE} = \frac{1}{n} \sum_{i=1}^{n} \left| y_i - \hat{y}_i \right|
$$

Where:

- $n$ is the total number of observations.
- $y_i$ is the actual value.
- $\hat{y}_i$ is the predicted value.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mae

### 8.2.3 Mean Squared Error (MSE)
Measures the average of the squares of the errors, giving more weight to larger errors
$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$
Where:
- $n$ is the total number of observations.
- $y_i$ is the actual value.
- $\hat{y}_i$ is the predicted value.

In [ ]:
mse = mean_squared_error(y_test, y_pred)
mse

### 8.2.4 Root Mean Squared Error (RMSE)
#### The square root of the Mean Squared Error, providing error metrics in the same units as the target variable.
$RMSE= \sqrt{MSE}$

In [ ]:
rmse = (mse)**0.5
rmse

### 8.2.5 Adjusted R-squared

Adjusted R-squared is a modified version of R-squared that adjusts for the number of predictors in the model. Unlike R-squared, it increases only when a new predictor improves the model more than would be expected by chance. Therefore, it is useful for comparing models with different numbers of features.

The formula for $Adjusted$ $R^2$ is:

$$
\text{Adjusted } R^2 = 1 - \left( \frac{1 - R^2}{n - 1} \times (n - p - 1) \right)
$$

Where:

- $R^2$ is the coefficient of determination.
- $n$ is the number of observations.
- $p$ is the number of predictors.

In [ ]:
n= len(y_test)
p = x_test.shape[1]
adj_r2 = 1 - (1 - r2) * (n-1) / (n-p-1)
print("Adjusted R-squared :",adj_r2)

## 8.3 Residual Analysis
#### 8.3.1 Calculating Residuals
Residuals are the differences between the observed values and the predicted values.
$\epsilon_i = y_i - \hat{y} $

In [ ]:
residuals = y_test - y_pred

#### 8.3.2 Residual vs Fitted Values Plot

In [ ]:
plt.figure(figsize=(8,4))
plt.scatter(y_pred, residuals)
plt.axhline(y=0, color='yellow', linestyle='--')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted Values')
plt.show()

The residuals vs fitted values plot shows that residuals are randomly scattered around zero without a clear pattern, indicating that the **linearity assumption is reasonably satisfied.** A slight variation in spread is observed at higher fitted values, suggesting **mild heteroscedasticity**, but **no severe model violation** is evident.

### 8.3.3 Histogram of Residuals

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(residuals, kde=True)
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Histogram of Residuals')
plt.show()

The histogram of residuals is approximately bell-shaped and centered around zero, indicating that the **residuals are reasonably normally distributed.**

A **slight right tail** is observed, suggesting the presence of **a few larger positive residuals.** However, the **deviation from normality is minor** and is **unlikely to have a significant impact** on the model’s overall performance or validity.

### 8.3.4 Q-Q Plot of Residuals
A Q-Q plot of residuals compares the quantiles of the residuals to the quantiles of a theoretical normal distribution. If the residuals are normally distributed, the points will lie close to the 45-degree reference line. Deviations from this line suggest deviations from normality, such as skewness or kurtosis.

In [ ]:
# Ensure residuals are in a 1-dimensional array
residuals = np.ravel(residuals)
# Q-Q Plot
plt.figure(figsize=(10, 6))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title('Q-Q Plot')
plt.show()

###8.3.5 Shapiro-Wilk Test for normality
The Shapiro-Wilk test is a hypothesis test that is applied to a sample with a null hypothesis that the sample has been generated from a normal distribution. If the p-value is low, we can reject such a null hypothesis and say that the sample has not been generated from a normal distribution.

In [ ]:
shapiro_test = stats.shapiro(residuals)
print(f"Shapiro-Wilk Test: Statistic={shapiro_test.statistic:.3f}, pvalue={shapiro_test.pvalue}")

Although the Shapiro–Wilk test returned a very small p-value, the sample size is large, making the test highly sensitive to minor deviations from normality. Visual inspection of the residual histogram indicates approximate normality with slight right skewness. Therefore, the normality assumption is considered reasonably satisfied.

### 8.4 Cross Validation Analysis
Cross-validation in linear regression assesses model performance by partitioning the dataset into multiple folds. Each fold serves as a test set while the remaining data is used for training. This process provides a robust estimate of the model’s generalization ability and helps identify potential overfitting or underfitting.

In [ ]:
from sklearn.model_selection import cross_val_score
# Perform cross-validation
cv_scores = cross_val_score(model, x_train, y_train, cv=5, scoring='r2')

In [ ]:
# cross-validation results
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())
print("Standard Deviation of CV Scores:", cv_scores.std())